<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####Requirement - Analysis Data Set

Prepare club bookings dataset for analysis
```
+----------+------------+---------------+-------------------+--------------+
|booking_id| member_name|  facility_name|         start_time|booking_amount|
+----------+------------+---------------+-------------------+--------------+
```

In [0]:
bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

club_bookings_df = (
    bookings_df.join(facilities_df, "facility_id")
            .join(members_df, "member_id", "left")
            .selectExpr("booking_id",
                        "case when member_id==0 then 'Guest Member' else concat_ws(' ', first_name, last_name) end as member_name",
                        "facility_name","start_time",
                        "case when member_id == 0 then slots * guest_cost else slots * member_cost end as booking_amount")
)

club_bookings_df.display()

booking_id,member_name,facility_name,start_time,booking_amount
0,Darren Smith,Table Tennis,2022-07-03T11:00:00.000Z,0.0
1,Darren Smith,Massage Room 1,2022-07-03T08:00:00.000Z,70.0
2,Guest Member,Squash Court,2022-07-03T18:00:00.000Z,35.0
3,Darren Smith,Snooker Table,2022-07-03T19:00:00.000Z,0.0
4,Darren Smith,Pool Table,2022-07-03T10:00:00.000Z,0.0
5,Darren Smith,Pool Table,2022-07-03T15:00:00.000Z,0.0
6,Tracy Smith,Tennis Court 1,2022-07-04T09:00:00.000Z,15.0
7,Tracy Smith,Tennis Court 1,2022-07-04T15:00:00.000Z,15.0
8,Tim Rownam,Massage Room 1,2022-07-04T13:30:00.000Z,70.0
9,Guest Member,Massage Room 1,2022-07-04T15:00:00.000Z,160.0


Q1. Who are the top 5 members by total booking amount?

Prepare a report as the following.
```
member_name     | total_booking_amount
---------------------------------------
Tim Rownam      | 6480
Tim Boothe      | 3644
Gerald Butters  | 3343
Burton Tracy    | 2953
David Jones     | 2651
```

1.1 Try aggregation using select or selectExpr

In [0]:
result_df = (
    club_bookings_df.where("member_name != 'Guest Member'")
            .groupBy("member_name")
            .selectExpr("member_name",
                        "sum(booking_amount) as total_booking_amount")
)

result_df.display()

---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
File <command-500853422959552>, line 4
      1 result_df = (
      2     club_bookings_df.where("member_name != 'Guest Member'")
      3             .groupBy("member_name")
----> 4             .selectExpr("member_name",
      5                         "sum(booking_amount) as total_booking_amount")
      6 )
      8 result_df.display()

AttributeError: 'GroupedData' object has no attribute 'selectExpr'

1.2 Try using agg() transformation

In [0]:
from pyspark.sql.functions import expr, col

result_df = (
    club_bookings_df.where("member_name != 'Guest Member'")
            .groupBy("member_name")
            .agg(expr("sum(booking_amount) as total_booking_amount"))
            .orderBy(col("total_booking_amount").desc())
            .limit(5)
)

result_df.display()

member_name,total_booking_amount
Tim Rownam,6480.0
Tim Boothe,3644.0
Gerald Butters,3343.0
Burton Tracy,2953.0
David Jones,2651.0


Q2. Who are the members having total booking amount > 2500?

In [0]:
from pyspark.sql.functions import expr, col

result_df = (
    club_bookings_df.where("member_name != 'Guest Member'")
            .groupBy("member_name")
            .agg(expr("sum(booking_amount) as total_booking_amount"))
            .where("total_booking_amount > 2500")
)

result_df.display()

member_name,total_booking_amount
Tim Rownam,6480.0
Tim Boothe,3644.0
Burton Tracy,2953.0
David Jones,2651.0
Gerald Butters,3343.0


Q3. Find member wise facility bookings for more than 2500?
```
+-----------+--------------+--------------------+
|member_name| facility_name|total_booking_amount|
+-----------+--------------+--------------------+
| Tim Boothe|Massage Room 1|              2660.0|
| Tim Rownam|Massage Room 1|              6160.0|
+-----------+--------------+--------------------+
```


In [0]:
from pyspark.sql.functions import expr, col

result_df = (
    club_bookings_df.where("member_name != 'Guest Member'")
            .groupBy("member_name", "facility_name")
            .agg(expr("sum(booking_amount) as total_booking_amount"))
            .where("total_booking_amount > 2500")
)

result_df.display()

member_name,facility_name,total_booking_amount
Tim Boothe,Massage Room 1,2660.0
Tim Rownam,Massage Room 1,6160.0


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>